# MSBD 566 – Small Task

**Name:** David Reyes  
**Course:** MSBD 566  
**Date:** October 20, 2026  

I trained a random forest using our BRFSS 2015 diabetes data and compared built-in importance with permutation importance on held-out data.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")
X = df.drop(columns="Diabetes_012")
y = (df["Diabetes_012"] > 0).astype(int)  # 1 = prediabetes or diabetes
print("Data shape:", df.shape)
print("Identical response rows:", int(df.duplicated().sum()))

Data shape: (253680, 22)
Identical response rows: 23899


## 1. Split And Train

I held out validation rows so the model would not train on the same rows used to measure permutation importance.

In [2]:
X_work, X_test, y_work, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_work, y_work, test_size=0.20, stratify=y_work, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
print("Validation AUC:", round(roc_auc_score(y_valid, rf.predict_proba(X_valid)[:, 1]), 4))

Validation AUC: 0.8245


## 2. Compare Feature Importance

Built-in importance measures how useful each feature was when splitting trees. Permutation importance measures how much the held-out AUC drops when a feature is shuffled.

In [3]:
built_in = pd.Series(rf.feature_importances_, index=X.columns)

# Use 5,000 held-out rows to make permutation faster
X_perm, _, y_perm, _ = train_test_split(
    X_valid, y_valid, train_size=5000, stratify=y_valid, random_state=42
)
permutation = permutation_importance(
    rf, X_perm, y_perm, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=1
)

results = pd.DataFrame({
    "Built-in": built_in,
    "Permutation AUC drop": permutation.importances_mean
})
results["Built-in rank"] = results["Built-in"].rank(ascending=False, method="min").astype(int)
results["Permutation rank"] = results["Permutation AUC drop"].rank(ascending=False, method="min").astype(int)
results["Rank difference"] = (results["Built-in rank"] - results["Permutation rank"]).abs()

display(results.sort_values("Rank difference", ascending=False).round(4))

,Built-in,Permutation AUC drop,Built-in rank,Permutation rank,Rank difference
PhysHlth,0.0360,-0.0005,8,21,13
CholCheck,0.0071,0.0018,16,7,9
HvyAlcoholConsump,0.0069,0.0013,17,9,8
Smoker,0.0102,-0.0001,14,20,6
Income,0.0458,0.0004,6,12,6
NoDocbcCost,0.0024,0.0002,20,14,6
MentHlth,0.0186,0.0001,11,16,5
Sex,0.0132,0.0015,12,8,4
HeartDiseaseorAttack,0.0218,0.0018,9,6,3
DiffWalk,0.0426,0.0010,7,10,3


## 3. One Disagreement

**PhysHlth** measures the number of days of poor physical health. The built-in method ranked it much higher than permutation importance. It may overlap with other health features, so shuffling it has little impact because the forest can still use related information. Built-in importance can also favor features with more possible split values.

In [4]:
print(results.loc["PhysHlth"].round(4))
print("\nCorrelations with PhysHlth:")
print(X_train[["PhysHlth", "GenHlth", "DiffWalk"]].corr(method="spearman")["PhysHlth"].round(3))

Built-in                 0.0360
Permutation AUC drop    -0.0005
Built-in rank            8.0000
Permutation rank        21.0000
Rank difference         13.0000
Name: PhysHlth, dtype: float64

Correlations with PhysHlth:
PhysHlth    1.000
GenHlth     0.451
DiffWalk    0.416
Name: PhysHlth, dtype: float64


## Summary

I trained a random forest on our BRFSS 2015 diabetes data. One feature where the two methods disagreed was **PhysHlth** (days of poor physical health). Built-in importance ranked it **8th of 21**, but permutation importance gave it a mean AUC drop of **-0.0005**, which is essentially zero. This may happen because other features, such as **GenHlth** and **DiffWalk**, already provide similar health information. So the forest used PhysHlth while building trees, but shuffling it barely changed the model’s AUC on held-out data.

**Repeated rows:** The data has 23,899 identical rows but no person ID, so I could not group the split by person.

**GitHub link:** https://github.com/reyesd23036/MSBD566-Small-Task-DR

**Data:** https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset